# Групповая часть

In [ ]:
import glob, subprocess, os, re, shutil

from Bio import SeqIO, Phylo
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import matplotlib.pyplot as plt
import pandas as pd

## 2.

In [ ]:
ORG_MAP = {
    "Bromodomain": "Sbovis",
    "Chromo": "Sintercalatum",
    "JmjC_histone_demethylase": "Sguineensis",
    "MBT": "Sjaponicum",
    "Shaematobium": "Shaematobium",
    "Smattheei": "Smattheei",
    "Smekongi": "Smekongi",
    "Spindale": "Spindale",
}

FAMILIES = [
    "Bromodomain",
    "Chromo",
    "DNA_methylase",
    "HDAC",
    "JmjC",
    "MBT",
    "PHD",
    "SIR2",
]


def norm(s):
    return re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")


for FAMILY in FAMILIES:
    faa_files = sorted(f for f in glob.glob(f"{FAMILY}_*.faa"))

    recs = []
    for f in faa_files:
        suffix = os.path.basename(f)[len(FAMILY) + 1 :].rsplit(".faa", 1)[0]
        org = ORG_MAP.get(norm(suffix), norm(suffix))
        for i, r in enumerate(SeqIO.parse(f, "fasta"), 1):
            r.id = f"{org}_{i}"
            r.name = r.id
            r.description = ""
            recs.append(r)

    merged = f"{FAMILY}_group.faa"
    SeqIO.write(recs, merged, "fasta")

    aln = f"{FAMILY}_group.aln"
    trim = f"{FAMILY}_group.trim.aln"
    tree = f"{FAMILY}_group.tree"

    subprocess.run(f"mafft --auto '{merged}' > '{aln}'", shell=True, check=True)
    subprocess.run(
        f"./trimal/source/trimal -in '{aln}' -out '{trim}' -automated1",
        shell=True,
        check=True,
    )
    ft = "FastTree" if shutil.which("FastTree") else "fasttree"
    subprocess.run(f"{ft} '{trim}' > '{tree}'", shell=True, check=True)

    t = Phylo.read(tree, "newick")
    t.ladderize()
    fig = plt.figure(figsize=(12, 24))
    ax = fig.add_subplot(111)
    Phylo.draw(
        t,
        axes=ax,
        do_show=False,
        branch_labels=lambda c: f"{c.confidence:.2f}"
        if c.confidence and c.confidence > 0.9
        else "",
    )
    plt.title(FAMILY)
    plt.savefig(f"{FAMILY}_group_tree.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

## 3.

In [ ]:
def org_from_name(path, prefix):
    tag = os.path.basename(path)[len(prefix):].rsplit(".csv", 1)[0]
    tag = re.sub(r"[^A-Za-z0-9]+", "_", tag).strip("_")
    return ORG_MAP.get(tag, tag)

def run_mafft(in_fa, out_aln):
    subprocess.run(f"mafft --auto '{in_fa}' > '{out_aln}'", shell=True, check=True)

def z_subseq(r):
    p = str(r.prom_seq).upper()
    if r.strand == "+":  i, j = r.z_start - r.prom_start, r.z_end - r.prom_start
    else:                i, j = r.prom_end - r.z_end,     r.prom_end - r.z_start
    return p[max(0,i):max(0,j)]

pid2fam = {}
for f in glob.glob("*.faa"):
    base = os.path.basename(f); fam = base.split("_")[0]
    if fam not in FAMILIES or base.endswith("_group.faa"): continue
    for rec in SeqIO.parse(f, "fasta"):
        pid2fam.setdefault(rec.id.split("|")[-1], fam)

In [ ]:
MIN_FOR_ALN = 3

recs_all, recs_epi, rows = [], [], []
for f in sorted(glob.glob("g4_promoters_*.csv")):
    org = org_from_name(f, "g4_promoters_")
    df = pd.read_csv(f)
    if df.empty or "g4_seq" not in df.columns:
        continue
    for r in df.itertuples():
        seq = str(r.g4_seq).upper()
        runs = [m.span() for m in re.finditer(r"G{3,5}", seq)]
        if not (3 <= len(runs) <= 8 and len(seq) <= 80):
            continue
        fam = pid2fam.get(r.protein_id)
        rid = f"{org}|{fam or 'NA'}|{r.protein_id}"
        rec = SeqRecord(Seq(seq), id=rid, description="")
        recs_all.append(rec)
        if fam:
            recs_epi.append(rec)
        rows.append(
            {
                "id": rid,
                "organism": org,
                "family": fam or "",
                "protein_id": r.protein_id,
                "is_epi": bool(fam),
                "length": len(seq),
                "n_Gruns": len(runs),
                "run_lengths": ",".join(str(e - s) for s, e in runs),
                "loop_lengths": ",".join(
                    str(runs[i + 1][0] - runs[i][1]) for i in range(len(runs) - 1)
                ),
                "seq": seq,
            }
        )

recs = recs_epi if len(recs_epi) >= MIN_FOR_ALN else recs_all

var = pd.DataFrame(rows)
var.to_csv("g4_variation.csv", index=False)
fa, aln = "g4_motifs_group.fasta", "g4_motifs_group.aln"
SeqIO.write(recs, fa, "fasta")
if len(recs) >= 2:
    run_mafft(fa, aln)

show = var[var.is_epi] if recs is recs_epi else var
print(
    show[
        ["organism", "family", "protein_id", "n_Gruns", "run_lengths", "loop_lengths"]
    ].to_string(index=False)
)

## 4.

In [ ]:
MIN_FOR_ALN = 3

recs_all, recs_epi, rows = [], [], []
for f in sorted(glob.glob("zdna_promoters_*.csv")):
    org = org_from_name(f, "zdna_promoters_")
    df = pd.read_csv(f)
    if df.empty or "prom_seq" not in df.columns:
        continue
    df = df.copy()
    df["z_len"] = df.z_end - df.z_start
    df = df.sort_values("z_len", ascending=False).drop_duplicates(subset=["protein_id"])
    for r in df.itertuples():
        seq = z_subseq(r)
        if not seq:
            continue
        fam = pid2fam.get(r.protein_id)
        rid = f"{org}|{fam or 'NA'}|{r.protein_id}"
        alt = sum(1 for a, b in zip(seq, seq[1:]) if (a in "AG") != (b in "AG"))
        rec = SeqRecord(Seq(seq), id=rid, description="")
        recs_all.append(rec)
        if fam:
            recs_epi.append(rec)
        rows.append(
            {
                "id": rid,
                "organism": org,
                "family": fam or "",
                "protein_id": r.protein_id,
                "is_epi": bool(fam),
                "z_len": int(r.z_len),
                "aln_len": len(seq),
                "alt_steps": alt,
                "GC%": round(100 * sum(c in "GC" for c in seq) / len(seq), 1),
                "seq": seq,
            }
        )

recs = recs_epi if len(recs_epi) >= MIN_FOR_ALN else recs_all

var = pd.DataFrame(rows)
var.to_csv("zdna_variation.csv", index=False)
fa, aln = "zdna_group.fasta", "zdna_group.aln"
SeqIO.write(recs, fa, "fasta")
if len(recs) >= 2:
    run_mafft(fa, aln)

show = var[var.is_epi] if recs is recs_epi else var
print(
    show[["organism", "family", "protein_id", "z_len", "alt_steps", "GC%"]].to_string(
        index=False
    )
)